# ShowPDF

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/electronmicroscopy/quantem.widget/blob/main/docs/tutorials/showpdf.ipynb)

`ShowPDF` computes the electron pair distribution function (ePDF) of a 4D-STEM scan with quantem's
`PairDistributionFunction`. On the scan image you select the positions to average; the curve panel
shows I(k) with its fitted background, F(k), G(r) or g(r) of that selection, and updates as you
change the settings.

This tutorial uses a simulated 4D-STEM scan of an amorphous tantalum model
([Ding, Asta, and Ritchie (2017), DOI 10.1073/pnas.1705723114](https://doi.org/10.1073/pnas.1705723114)),
so the result can be compared with the model's own g(r). ShowPDF needs `PairDistributionFunction`
from quantem pull request 248, which is on quantem's dev branch until the next quantem release.

In [ ]:
# @title Install QuantEM { display-mode: "form" }
if get_ipython().__class__.__module__.startswith("google.colab"):
    !wget -q https://raw.githubusercontent.com/electronmicroscopy/quantem.widget/main/scripts/install_colab.py -O install_quantem.py
    %run install_quantem.py

from quantem.widget import ShowPDF

## Load the simulated scan

`showpdf_ta` downloads the scan (about 150 MB, cached after the first call) and returns it
calibrated: a 20 × 20 scan of 307 × 307 patterns, with k in 1/Å (k = 1/d).
`showpdf_ta_ground_truth` returns the g(r) of the atomic model.

In [ ]:
from quantem.widget.datasets import showpdf_ta, showpdf_ta_ground_truth

scan = showpdf_ta()
r_model, g_model = showpdf_ta_ground_truth()

## Number density

ShowPDF never estimates the number density ρ₀; g(r) and the low-r correction of G(r) need it.
The simulated model holds 1,024,000 atoms in a cube of side 268.6053 Å.

In [ ]:
rho0 = 1_024_000 / 268.6053**3  # atoms/Å³
rho0

## The whole scan

With nothing selected, ShowPDF averages every scan position. The background fit starts at
k = 0.05 1/Å, and G(r) runs to 20 Å.

The low-r correction makes G(r) follow −4πρ₀r from 0 to `r_cut`, a range with no atom pairs, so
`r_cut` must stay below the nearest-neighbour distance. The model's g(r) is zero below 2.4 Å,
so this tutorial uses 2.0 Å; the default, 0.8 Å, leaves a false peak near 1 Å here.

The buttons above the curve switch between I(k), F(k), G(r) and g(r), and the sliders below it
change the fit range, the k window, the filters and the correction.

In [ ]:
widget = ShowPDF(
    scan,
    k_min_fit=0.05,
    r_max=20.0,
    density_value=rho0,
    damp_origin_oscillations=True,
    r_cut=2.0,
    plot_mode="gr",
)
widget

## Compare with the atomic model

`widget.pdf` is the `PairDistributionFunction` behind the widget, holding the curves it shows.
The peaks of the whole-scan g(r) sit where the model's do. They are lower and broader, partly
because the patterns end at k = 2.26 1/Å, where the window takes F(k) to zero.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(r_model, g_model, color="black", linewidth=3, label="atomic model")
ax.plot(widget.pdf.r, widget.pdf.pdf, color="tab:red", label="ShowPDF, whole scan")
ax.set_xlabel("r (Å)")
ax.set_ylabel("g(r)")
ax.legend()
plt.show()

## A line scan

Draw a line on the scan image, or set one with `set_line`, and switch from avg to scan
(`line_mode="linescan"`): ShowPDF computes one curve per position along the line and shows them
as a heatmap or as stacked curves (`linescan_view="waterfall"`). The band around the line is
`width` scan pixels wide, an odd number.

In [ ]:
line_scan = ShowPDF(
    scan,
    k_min_fit=0.05,
    r_max=20.0,
    density_value=rho0,
    damp_origin_oscillations=True,
    r_cut=2.0,
    plot_mode="gr",
    line_mode="linescan",
    linescan_view="waterfall",
)
line_scan.set_line(3, 2, 15, 11, width=3)
line_scan

## Save and share

`summary()` prints the selection and every setting. `save_results` writes the curves as CSV
files with the settings in one zip, and `save_image` saves the shown curve as an image.
`export_html` writes a standalone page; with `encoding="uint8"`, changing the selection or a
setting in that page recomputes the curves, without Python.

In [ ]:
widget.summary()
widget.save_results("ta_pdf_results.zip", overwrite=True)
widget.save_image("ta_gr.png")
widget.export_html("ta_pdf.html", encoding="uint8")